# TP1 — Observations already on Kabre: maps, time series, vertical sections, and a first look at NCO/CDO

**Suggested duration: ~3h** · **Prerequisites**: Course 1 (observation products) and Course 2 (anatomy of a netCDF file).

This morning you saw *what* observation products are and *how* a netCDF file is organised. This afternoon you open real files, already stored on Kabre, and turn them into figures and numbers, first with **Python (xarray)**, then with two command-line toolboxes, **NCO** and **CDO**.

## Learning objectives
By the end of this TP you should be able to:
1. Inspect a netCDF file (dimensions, variables, units) from Python and from the command line.
2. Plot a 2D map, a time series at a point, and a vertical section.
3. Extract a sub-region and compute a time mean with NCO and with CDO.
4. **Cross-check** a result by computing it with independent methods.
5. Read an error message and fix a broken cell.

## How to work through this notebook
- Cells marked `# TODO` contain code **you need to complete** (replace the `...` or `___`). Everything else is already written.
- After each exercise, a **✅ Check** box tells you what a correct result should look like — use it to self-correct before moving on.
- **🚀 Advanced track** blocks are optional: do them only if you finished the main exercise.
- Stuck on a `# TODO`? Below each exercise, a **💡 Hint** box (and, as a last resort, a **🔑 Solution** box) can be opened with a click. Try for a few minutes before opening them!
- Stuck for more than 10 minutes? Ask a trainer — that's what we're here for.

---
## 0. Setup — imports, paths and helper function

Run this cell as is. It defines:
- `OBS_DIR`: where the observation files are stored on Kabre (read-only, shared by everyone).
- `SST_FILE`, `WOA_FILE`: the two files used today.
- `WORK_DIR`: a folder `data_tp1/`, created next to this notebook, where **your** output files go.
- `LON_MIN, LON_MAX, LAT_MIN, LAT_MAX`: our study box (Pacific coast of Costa Rica).
- `to_celsius(da)`: converts a temperature to °C **if** its `units` attribute says Kelvin.

In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Observation files already on Kabre (read-only)
OBS_DIR = "/data0/user/gcambon/DATA/DATASETS_PSF"
if not Path(OBS_DIR).exists():          # not on Kabre: instructor's local copy
    OBS_DIR = "../data/DATASETS_PSF"

SST_FILE = f"{OBS_DIR}/COSTARICA_DS/sst_central_america_2023.nc"
WOA_FILE = f"{OBS_DIR}/GLOBAL_DS/WOA2009/temp_seas.cdf"

# Your own output files go here
WORK_DIR = Path("data_tp1")
WORK_DIR.mkdir(exist_ok=True)

# Study box: Pacific coast of Costa Rica
LON_MIN, LON_MAX = -90.0, -80.0
LAT_MIN, LAT_MAX = 5.0, 15.0


def to_celsius(da):
    """Return `da` in °C if its 'units' attribute says Kelvin, unchanged otherwise."""
    if da.attrs.get("units", "").lower() in ("k", "kelvin", "degk", "deg_k"):
        out = da - 273.15
        out.attrs = {**da.attrs, "units": "°C"}
        return out
    return da


!ls -lh {OBS_DIR}/CAMERICA_DS {OBS_DIR}/GLOBAL_DS/WOA2009

**✅ Check**: the `ls` lists (among others) `sst_central_america_2023.nc` (≈ 600 MB) and `temp_seas.cdf`. If you get *No such file or directory*, check that you are on Kabre and that `OBS_DIR` is not commented out.

---
## 1. Anatomy of a real file (applying Course 2)

Let's open the satellite SST file (OSTIA, daily, 0.05°) and find the elements seen in Course 2: **dimensions**, **coordinates**, **variables**, **attributes**.

In [ ]:
ds_sst = xr.open_dataset(SST_FILE)
ds_sst

**🎯 Your turn**: print the key facts of this file. Hints: `ds_sst.sizes`, `list(ds_sst.data_vars)`, `ds_sst[VAR].attrs`, `ds_sst.time.values[0]`.

In [ ]:
# TODO: replace each ... with the right expression
print("Dimensions and sizes :", ...)
print("Data variables       :", ...)
VAR_SST = ...   # the name of the SST variable, as a string
print("SST units            :", ...)
print("First / last date    :", ..., "/", ...)
print("Latitude increasing (south → north)?", ...)

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `ds_sst.sizes` · `list(ds_sst.data_vars)` · `ds_sst[VAR_SST].attrs["units"]`
- `ds_sst.time.values[0]` and `[-1]` · compare `ds_sst.latitude[0]` with `ds_sst.latitude[-1]`

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
print("Dimensions and sizes :", dict(ds_sst.sizes))
print("Data variables       :", list(ds_sst.data_vars))
VAR_SST = "analysed_sst"
print("SST units            :", ds_sst[VAR_SST].attrs["units"])
print("First / last date    :", ds_sst.time.values[0], "/", ds_sst.time.values[-1])
print("Latitude increasing (south → north)?", bool(ds_sst.latitude[0] < ds_sst.latitude[-1]))
```

</details>

**✅ Check**:
- dimensions `time: 365`, `latitude: 700`, `longitude: 1200` (0.05° grid, 120°W–60°W, 5°S–30°N);
- one data variable, `analysed_sst`, in **kelvin** → we will need `to_celsius()`;
- dates from 2023-01-01 to 2023-12-31 (daily);
- latitude increasing: `True`. Remember this for any `.sel(latitude=slice(a, b))` later on.

---
# Part A — Python / xarray

## A.1 A 2D horizontal map: annual mean SST

The typical xarray recipe is **select → reduce → plot**. Here: take the SST variable, convert it to °C, average it over `time`, and map it.

> ⏳ Averaging 365 daily maps reads the whole file (~2 GB in memory): this cell can take up to a minute.

In [ ]:
# TODO: convert the SST to °C, then average over the time dimension
sst_mean = ...

fig, ax = plt.subplots(figsize=(10, 6), subplot_kw={"projection": ccrs.PlateCarree()})
sst_mean.plot(ax=ax, transform=ccrs.PlateCarree(), cmap="turbo",
              cbar_kwargs={"label": "SST (°C)"})
ax.add_feature(cfeature.LAND, facecolor="0.8", zorder=2)
ax.add_feature(cfeature.COASTLINE, zorder=3)
ax.gridlines(draw_labels=True, linewidth=0.3)
ax.set_title("Mean SST 2023 — OSTIA")
plt.show()

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- Two steps chained: `to_celsius(ds_sst[VAR_SST])` gives °C, then `.mean(dim="time")`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
sst_mean = to_celsius(ds_sst[VAR_SST]).mean(dim="time")
```

</details>

**✅ Check**: values between roughly **16 °C and 31 °C**. Warmest (> 29 °C) in the Eastern Pacific warm pool off southern Mexico/Guatemala and in the Panama Bight; coldest in the north-west corner (California Current, off Baja California) and, to a lesser extent, south of the Equator near Ecuador. If your colorbar goes up to ~300, you forgot `to_celsius()`.

## A.2 A time series at a point

We look at the **Gulf of Papagayo** (north-west Costa Rica), where strong trade-wind jets blow across the isthmus in winter.

**🎯 Your turn**: select the SST time series at the grid point nearest to (87°W, 10°N). Hint: `.sel(..., method="nearest")`.

In [ ]:
lon0, lat0 = -87.0, 10.0   # Gulf of Papagayo

# TODO: select the SST at the nearest grid point, in °C
ts = ...

fig, ax = plt.subplots(figsize=(12, 4))
ts.plot(ax=ax)
ax.set_ylabel("SST (°C)")
ax.set_title(f"Daily SST at ({lon0}°E, {lat0}°N) — 2023")
ax.grid(alpha=0.3)
plt.show()

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `.sel(longitude=..., latitude=..., method="nearest")` on the SST converted to °C.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
ts = to_celsius(ds_sst[VAR_SST]).sel(longitude=lon0, latitude=lat0, method="nearest")
```

</details>

**✅ Check**: a clear seasonal cycle, with **sharp cold events down to ~24 °C in January-February** (Papagayo wind jets mix the surface layer and bring up colder water) and a **maximum around 31 °C in May**, then a plateau near 29.5-30 °C until October. Monthly means go from ~25.5 °C (February) to ~30.6 °C (May).

💬 **Think**: would you see the same cycle at the Caribbean coast, e.g. (82°W, 12°N)? Try it by changing `lon0, lat0`.

## A.3 A vertical section: World Ocean Atlas climatology

`temp_seas.cdf` is a **3D climatology** (Course 1): long-term mean temperature, on a 1° grid, 33 depth levels, for **4 seasons**. Its dimension names are **not** the usual ones:

| Usual name | In this file | Values |
|---|---|---|
| longitude | `X` | −179.5 … 179.5 |
| latitude | `Y` | −89.5 … 89.5 |
| depth | `Z` | 0 … 5500 m |
| time | `T` | 4 seasons (1.5, 4.5, 7.5, 10.5 = mid-season months) |

So before plotting a single section we must **average over the 4 seasons** (`T`).

**🎯 Your turn**: build the annual-mean temperature section along 9.5°N, for the Eastern Pacific (120°W–84°W) and the upper 500 m.

In [ ]:
ds_woa = xr.open_dataset(WOA_FILE)
print(ds_woa)

lat_section = 9.5

# TODO: 1) average "temperature" over the season dimension
#       2) select the latitude nearest to lat_section
#       3) keep X between -120 and -84 and Z between 0 and 500
temp_section = ...

fig, ax = plt.subplots(figsize=(11, 5))
temp_section.plot.contourf(x="X", y="Z", ax=ax, levels=np.arange(8, 30, 1), cmap="turbo",
                           cbar_kwargs={"label": "Temperature (°C)"})
temp_section.plot.contour(x="X", y="Z", ax=ax, levels=[20], colors="k", linewidths=1.5)
ax.invert_yaxis()
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Depth (m)")
ax.set_title(f"WOA2009 annual mean temperature — {lat_section}°N (black line: 20 °C isotherm)")
plt.show()

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- Chain three steps: `ds_woa["temperature"].mean(dim="T")`, then `.sel(Y=lat_section, method="nearest")`, then `.sel(X=slice(...), Z=slice(...))`.
- `method="nearest"` cannot be mixed with `slice` in the same `.sel()`: that's why there are two.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
temp_section = (
    ds_woa["temperature"]
    .mean(dim="T")
    .sel(Y=lat_section, method="nearest")
    .sel(X=slice(-120, -84), Z=slice(0, 500))
)
```

</details>

**✅ Check**: a warm surface layer (> 26 °C) over a sharp **thermocline** (tight colour gradient). The 20 °C isotherm (a common marker of the thermocline) rises from **~70 m at 120°W to ~30-35 m east of 90°W**: the thermocline shoals towards the **Costa Rica Dome**, a region of open-ocean upwelling centred near 9°N, 90°W. The white strip on the right is land (Costa Rica on the 1° WOA grid).

⚠️ If you get `KeyError: 'lat'` or `'depth'`, go back to the table above: the dimensions are `X/Y/Z/T`.

### 🚀 Advanced track A — the seasonal cycle of the Costa Rica Dome

Instead of averaging over `T`, plot the **4 seasonal sections** along 9.5°N in a 2×2 figure with a single shared colorbar. In which season is the Dome shallowest? Does it match the timing of the Papagayo cold events seen in A.2?

In [ ]:
# 🚀 Your code here (hint: loop over ds_woa.T.values and use .sel(T=...))

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- `for ax, t in zip(axes.flat, ds_woa.T.values):` then `.sel(T=t)` instead of `.mean(dim="T")`.
- `add_colorbar=False` in each panel, then a single `fig.colorbar(cf, ax=axes)`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
season_names = {1.5: "JFM", 4.5: "AMJ", 7.5: "JAS", 10.5: "OND"}

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True, sharey=True)
for ax, t in zip(axes.flat, ds_woa.T.values):
    sec = (ds_woa["temperature"].sel(T=t)
           .sel(Y=lat_section, method="nearest")
           .sel(X=slice(-120, -84), Z=slice(0, 500)))
    cf = sec.plot.contourf(x="X", y="Z", ax=ax, levels=np.arange(8, 30, 1), cmap="turbo",
                           add_colorbar=False)
    sec.plot.contour(x="X", y="Z", ax=ax, levels=[20], colors="k", linewidths=1.5)
    ax.set_title(season_names[float(t)])
    ax.set_xlabel("")
    ax.set_ylabel("")
axes[0, 0].invert_yaxis()   # axes are shared: inverting one inverts all
fig.colorbar(cf, ax=axes, label="Temperature (°C)", shrink=0.8)
fig.suptitle(f"WOA2009 seasonal temperature sections — {lat_section}°N")
plt.show()

# Answer: the 20 °C isotherm forms a clear dome, ~35 m deep near 90-92°W, in JAS
# (and still in OND): the Dome is strongest in the second half of the year. In JFM the
# thermocline is shallow only right at the coast. The Papagayo cold events (A.2) peak in
# JFM: they are a *coastal* wind-driven process, distinct from the Dome, although the
# winter wind jets also help "seed" the Dome.
```

</details>

---
# Part B — Command line: NCO and CDO

## B.0 Why the command line?

On an HPC cluster you often handle files too large to load into memory, or hundreds of files to process in a batch script (SLURM job). **NCO** (netCDF Operators) and **CDO** (Climate Data Operators) are made for that: one command, one operation, no Python code, and very fast.

In a Jupyter cell, a line starting with `!` is sent to the shell. Python variables can be inserted with **single braces**: `!ncdump -h {SST_FILE}`.

## B.1 NCO: `ncdump`, `ncks`, `ncwa`

### 📋 NCO cheat-sheet

| Task | Command |
|---|---|
| Show the header (dimensions, variables, attributes) | `ncdump -h in.nc` |
| Print the values of one variable | `ncdump -v VAR in.nc` |
| Cut a sub-region (by **coordinate values**) | `ncks -O -d DIM,MIN,MAX -d DIM2,MIN,MAX in.nc out.nc` |
| Keep only some variables | `ncks -O -v VAR1,VAR2 in.nc out.nc` |
| Average over a dimension | `ncwa -O -a DIM in.nc out.nc` |

- `-O` = **overwrite** the output if it already exists (otherwise re-running the cell fails or asks a question).
- ⚠️ In `-d DIM,MIN,MAX`, write the bounds **with a decimal point** (`-90.0`): without it (`-90`), NCO interprets them as **indices** (grid-point numbers), not coordinate values!

In [ ]:
!ncdump -h {SST_FILE}

**🎯 Your turn**: cut the study box (`LON_MIN…LON_MAX`, `LAT_MIN…LAT_MAX`) out of `SST_FILE` into `data_tp1/sst_costarica.nc`, then average it over time into `data_tp1/sst_costarica_mean_nco.nc`.

In [ ]:
BOX_NCO = WORK_DIR / "sst_costarica.nc"
MEAN_NCO = WORK_DIR / "sst_costarica_mean_nco.nc"

# TODO: complete the dimension names and bounds (with a decimal point!)
!ncks -O -d ___,{LON_MIN},{LON_MAX} -d ___,___,___ {SST_FILE} {BOX_NCO}
!ncdump -h {BOX_NCO} | head -15

# TODO: complete the operator and the dimension to average over
!___ -O -a ___ {BOX_NCO} {MEAN_NCO}
!ncdump -h {MEAN_NCO} | head -15

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- Dimension names are in the `ncdump -h` output above: `longitude`, `latitude`.
- Average over a dimension: `ncwa -O -a DIM in.nc out.nc` (see the cheat-sheet).

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
!ncks -O -d longitude,{LON_MIN},{LON_MAX} -d latitude,{LAT_MIN},{LAT_MAX} {SST_FILE} {BOX_NCO}
!ncwa -O -a time {BOX_NCO} {MEAN_NCO}
```

</details>

**✅ Check**:
- `sst_costarica.nc`: `time = 365`, `latitude = 200`, `longitude = 200` (10° / 0.05°).
- `sst_costarica_mean_nco.nc`: **no `time` dimension any more** (it has been averaged out), still 200 × 200.

Since `LON_MIN` etc. are Python floats (`-90.0`), `{LON_MIN}` is written with a decimal point — good. If you get *"dimension … not in input file"*, check the exact names in the `ncdump -h` output.

💡 Why `ncwa` and not `ncra`? `ncra` (*record average*) only averages over the **record** (unlimited) dimension, and `time` is not one in this file. `ncwa -a DIM` averages over **any** dimension.

## B.2 CDO: `sinfo`, `sellonlatbox`, `timmean`

### 📋 CDO cheat-sheet

| Task | Command |
|---|---|
| Short summary of a file | `cdo sinfo in.nc` |
| Cut a lon/lat box | `cdo -O sellonlatbox,LON1,LON2,LAT1,LAT2 in.nc out.nc` |
| Time mean (whole period) | `cdo -O timmean in.nc out.nc` |
| Monthly means | `cdo -O monmean in.nc out.nc` |
| Spatial (area-weighted) mean | `cdo -O fldmean in.nc out.nc` |
| **Chaining** (right to left, no intermediate file) | `cdo -O timmean -sellonlatbox,LON1,LON2,LAT1,LAT2 in.nc out.nc` |

Note the order of arguments: CDO takes **lon, lon, lat, lat**, without dimension names.

In [ ]:
!cdo sinfo {SST_FILE}

**🎯 Your turn**: do the same as in B.1 with CDO: cut the box into `data_tp1/sst_costarica_cdo.nc`, then compute the time mean into `data_tp1/sst_costarica_mean_cdo.nc`.

In [ ]:
BOX_CDO = WORK_DIR / "sst_costarica_cdo.nc"
MEAN_CDO = WORK_DIR / "sst_costarica_mean_cdo.nc"

# TODO: complete the operator arguments
!cdo -O sellonlatbox,___,___,___,___ {SST_FILE} {BOX_CDO}

# TODO: complete the operator
!cdo -O ___ {BOX_CDO} {MEAN_CDO}
!cdo sinfo {MEAN_CDO}

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- CDO order is lon, lon, lat, lat: `{LON_MIN},{LON_MAX},{LAT_MIN},{LAT_MAX}`.
- Time mean over the whole period: see the cheat-sheet.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
!cdo -O sellonlatbox,{LON_MIN},{LON_MAX},{LAT_MIN},{LAT_MAX} {SST_FILE} {BOX_CDO}
!cdo -O timmean {BOX_CDO} {MEAN_CDO}
```

</details>

**✅ Check**: `cdo sinfo` on the result shows a grid of **200 × 200** points (`points=40000 (200x200)`), and **1 time step** (dated 2023-07-02, the middle of the period).

CDO prints `cdi warning … Variable not found - >lon<` (and, when chaining, sometimes a long `HDF5-DIAG` message): the file's metadata mentions `lon lat` while the coordinates are called `longitude latitude`. These are **warnings, not errors** — what matters is that the output file is created and correct.

### 🚀 Advanced track B — chaining CDO operators, and a subtle difference

1. In **one** CDO command (chaining, no intermediate file), compute the **monthly mean SST averaged over the study box**, from `SST_FILE` into `data_tp1/sst_box_monthly_cdo.nc` (hint: `fldmean`, `monmean`, `sellonlatbox`).
2. Compute the same 12 values in Python with xarray (hint: `.resample(time="1MS").mean()` then `.mean(["latitude", "longitude"])`) and compare them.
3. The two series differ slightly. Why? Read the CDO documentation of `fldmean`, then fix your xarray calculation so they agree (hint: `.weighted(...)`).

In [ ]:
# 🚀 Your code here

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- CDO chain, read right to left: `fldmean -monmean -sellonlatbox,...`.
- Area weights on a lon/lat grid: `np.cos(np.deg2rad(box.latitude))`.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
MONTHLY_CDO = WORK_DIR / "sst_box_monthly_cdo.nc"
!cdo -O fldmean -monmean -sellonlatbox,{LON_MIN},{LON_MAX},{LAT_MIN},{LAT_MAX} {SST_FILE} {MONTHLY_CDO}

cdo_series = to_celsius(xr.open_dataset(MONTHLY_CDO)[VAR_SST]).squeeze()

box = to_celsius(ds_sst[VAR_SST]).sel(longitude=slice(LON_MIN, LON_MAX),
                                      latitude=slice(LAT_MIN, LAT_MAX))
box_monthly = box.resample(time="1MS").mean()
xr_plain = box_monthly.mean(["latitude", "longitude"])

# fldmean weights each grid cell by its area, i.e. by cos(latitude)
weights = np.cos(np.deg2rad(box.latitude))
xr_weighted = box_monthly.weighted(weights).mean(["latitude", "longitude"])

print("month   CDO fldmean   xarray plain   xarray weighted")
for m in range(12):
    print(f"{m + 1:5d}   {float(cdo_series[m]):11.3f}   {float(xr_plain[m]):12.3f}   {float(xr_weighted[m]):15.3f}")

# Answer: CDO fldmean is area-weighted (cells get smaller towards the poles, ~cos(lat)),
# a plain xarray .mean() gives every grid point the same weight. The difference is tiny
# here (a few thousandths of °C over 5-15°N) but grows for big or high-latitude boxes.
# With the cos(lat) weights, xarray and CDO agree to 0.001 °C.
```

</details>

---
## 2. Cross-check: three methods, one number

A good habit in science: **never trust a single calculation**. You have computed the 2023 mean SST over the box in three independent ways (xarray, NCO, CDO). Do they agree?

**🎯 Your turn**: compute the xarray value (box selection + mean over everything), and read the NCO and CDO results.

In [ ]:
# NCO and CDO: open the output files and average the remaining 2D map
val_nco = float(to_celsius(xr.open_dataset(MEAN_NCO)[VAR_SST]).mean())
val_cdo = float(to_celsius(xr.open_dataset(MEAN_CDO)[VAR_SST]).mean())

# TODO: select the box in ds_sst (latitude is increasing, see section 1), convert to °C, average
val_xr = ...

for name, v in [("Python/xarray", val_xr), ("NCO", val_nco), ("CDO", val_cdo)]:
    print(f"{name:15s} : {v:.3f} °C")

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- Same box as NCO/CDO: `ds_sst[VAR_SST].sel(longitude=slice(LON_MIN, LON_MAX), latitude=slice(LAT_MIN, LAT_MAX))`.
- Then `to_celsius(...)`, `.mean()`, and `float(...)` to get a plain number.

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
sst_box = ds_sst[VAR_SST].sel(longitude=slice(LON_MIN, LON_MAX),
                              latitude=slice(LAT_MIN, LAT_MAX))
val_xr = float(to_celsius(sst_box).mean())
```

</details>

**✅ Check**: the three values agree to within **~0.001 °C**, around **29.04 °C**.

💬 If one of them is `nan`, or clearly different: which selection is wrong? Check the box bounds, the latitude order, the units. Finding *why* two methods disagree is often where you learn the most.

---
## 3. 🐞 Debugging exercise

A colleague sends you this cell: *"it should print the 2023 mean SST over the study box, but it doesn't work"*. It contains **three bugs**. Two raise an error, one is **silent** (no error, but a wrong result).

Run it, **read the error message** (the last line is usually the most useful), fix one bug, run again… until it prints the same value as `val_xr` in section 2.

In [ ]:
# 🐞 Fix me! (3 bugs)
sst_box_bug = ds_sst["sea_surface_temperature"].sel(lon=slice(-90, -80), latitude=slice(15, 5))
sst_box_bug = to_celsius(sst_box_bug)
print(f"Box mean SST: {float(sst_box_bug.mean()):.3f} °C")

<details><summary>💡 <b>Hint</b> (try for a few minutes first!)</summary>

- Read the **last line** of each error message.
- Check `list(ds_sst.data_vars)`, `ds_sst.dims`, and the order of the latitudes (section 1).

</details>

<details><summary>🔑 <b>Solution</b> (only if you are really stuck)</summary>

```python
# Fixed version
sst_box_bug = ds_sst["analysed_sst"].sel(longitude=slice(-90, -80), latitude=slice(5, 15))

# Bug 1: KeyError 'sea_surface_temperature'  -> the variable is called analysed_sst
#        (check with list(ds_sst.data_vars)).
# Bug 2: KeyError / ValueError on 'lon'        -> the dimension is called longitude.
# Bug 3 (silent): latitude is stored increasing, so slice(15, 5) selects nothing;
#        the mean of an empty array is nan (with only a RuntimeWarning).
```

</details>

**✅ Check**: the fixed cell prints the same value as `val_xr` (≈ 29.04 °C). If you get `nan` without any error, you have not found the silent bug yet: look at the order of the latitudes (section 1).

---
## 4. NCO/CDO vs Python: when to use what?

| Criterion | NCO / CDO | Python (xarray) |
|---|---|---|
| Batch processing of many files, SLURM scripts | ✅ Very well suited | Possible, more verbose |
| Large files (avoid loading everything into memory) | ✅ Optimised | Needs dask to stay efficient |
| Maps and figures | ❌ Not designed for this | ✅ Very rich (matplotlib, cartopy) |
| Advanced statistics, machine learning | ❌ | ✅ |
| Learning curve | Quick for simple operations | Longer at first |
| Reproducibility / sharing | A script of commands | ✅ Self-documenting notebook |

**In practice**: many people use CDO/NCO for the upstream pre-processing (extraction, averaging, regridding), then Python for the detailed analysis and the figures.

---
## 5. In pairs: apply it to another variable

Pick **one** of these files (in `OBS_DIR`), and with your partner:

| File | What |
|---|---|
| `CAMERICA_DS/ssh_central_america_2023.nc` | Sea surface height (satellite altimetry), daily 2023 |
| `CAMERICA_DS/sst_central_america_2024.nc` | Same SST as today, for 2024 (El Niño year!) |
| `GLOBAL_DS/WOA2009/salt_seas.cdf` | Salinity climatology (same structure as `temp_seas.cdf`) |
| `GLOBAL_DS/WOA2009/o2_seas.cdf` | Dissolved oxygen climatology |

1. Find the variable name, dimensions and units (`ncdump -h`).
2. Make **one map** and **one time series or section** with a sensible colorbar and units.
3. Compute **one mean value by two different methods** (e.g. xarray and CDO) and check that they agree.
4. Write 2-3 sentences: what do you see, and what physical process could explain it?

In [ ]:
# Your code here

---
## To go further

- xarray documentation: <https://docs.xarray.dev>
- NCO documentation: <https://nco.sourceforge.net/nco.html>
- CDO documentation and reference card: <https://code.mpimet.mpg.de/projects/cdo>
- Next TP (TP2): instead of using files already on Kabre, you will **download your own data** from Copernicus Marine.